# TCRP · MIT-BIH Arrhythmia Classification (EXP-C02)

Trains **TCRPClassifier** on the MIT-BIH pre-segmented heartbeat dataset (Kachuee et al.):
- **T = 187** samples per heartbeat
- **C = 5** AAMI classes: N (normal), S (supraventricular), V (ventricular), F (fusion), Q (unknown)
- Heavily imbalanced (N ≈ 90 %) — uses `WeightedRandomSampler` + weighted cross-entropy
- Optional adversarial (GRL) variant

**Runtime:** ~10 min on T4 GPU (150 epochs, early stopping usually kicks in at ~60-80 epochs).

**Notebook structure**
1. Environment setup
2. Download MIT-BIH data from Kaggle
3. Quick EDA
4. Train
5. Evaluate (accuracy, F1, confusion matrix)
6. Concept profiles per arrhythmia class
7. (Optional) Adversarial training

## 1 · Environment setup

In [ ]:
# Clone the TCRP repository
import os

REPO = "/content/TCRP"
if not os.path.isdir(REPO):
    !git clone https://github.com/AmTuTi1999/TCRP.git {REPO}
else:
    print("Repo already present — pulling latest")
    !git -C {REPO} pull

In [ ]:
# Install extra dependencies not bundled with Colab
!pip install -q hydra-core omegaconf kaggle

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, REPO)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"Repo    : {REPO}")

## 2 · Download MIT-BIH data from Kaggle

Data source: [Kaggle — ECG Heartbeat Categorization Dataset](https://www.kaggle.com/datasets/shayanfazeli/heartbeat)

**One-time setup:**
1. Go to kaggle.com → Account → Create API token → download `kaggle.json`
2. Upload `kaggle.json` when prompted below, **or** mount Google Drive if you already have it there.

In [ ]:
# Upload your kaggle.json API token
# Skip this cell if kaggle.json is already present at ~/.kaggle/kaggle.json

kaggle_cfg = Path("/root/.kaggle/kaggle.json")
if not kaggle_cfg.exists():
    from google.colab import files
    print("Please upload your kaggle.json …")
    uploaded = files.upload()
    kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
    kaggle_cfg.write_bytes(list(uploaded.values())[0])
    kaggle_cfg.chmod(0o600)
    print("✓ kaggle.json saved")
else:
    print("✓ kaggle.json already present")

In [ ]:
# Download and place files where MITBIHDataset expects them
# Path: TCRP/tcrp/data/raw/CRWU/mitbih_{train,test}.csv

DATA_ROOT = Path(REPO) / "tcrp" / "data" / "raw" / "CRWU"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

train_csv = DATA_ROOT / "mitbih_train.csv"
test_csv  = DATA_ROOT / "mitbih_test.csv"

if train_csv.exists() and test_csv.exists():
    print(f"✓ Data already present at {DATA_ROOT}")
else:
    print("Downloading from Kaggle …")
    import subprocess, zipfile, tempfile

    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "shayanfazeli/heartbeat",
             "--path", tmp, "--unzip"],
            check=True
        )
        for fname in ["mitbih_train.csv", "mitbih_test.csv"]:
            src = Path(tmp) / fname
            if src.exists():
                import shutil
                shutil.copy(src, DATA_ROOT / fname)
                print(f"  copied {fname}")

print(f"train : {train_csv}  exists={train_csv.exists()}")
print(f"test  : {test_csv}   exists={test_csv.exists()}")

## 3 · Quick EDA

In [ ]:
CLASS_NAMES = ["N — Normal", "S — Supraventricular", "V — Ventricular", "F — Fusion", "Q — Unknown"]
CLASS_COLORS = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b2"]

df_train = pd.read_csv(train_csv, header=None)
df_test  = pd.read_csv(test_csv,  header=None)

X_train = df_train.iloc[:, :-1].values
y_train = df_train.iloc[:, -1].values.astype(int)
X_test  = df_test.iloc[:,  :-1].values
y_test  = df_test.iloc[:,  -1].values.astype(int)

print(f"Train : {X_train.shape}  Test : {X_test.shape}")
print(f"Signal length (T) : {X_train.shape[1]}")
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution
ax = axes[0]
counts = np.bincount(y_train, minlength=5)
bars = ax.bar(range(5), counts, color=CLASS_COLORS, edgecolor="white", linewidth=0.5)
ax.set_xticks(range(5))
ax.set_xticklabels([n.split(" — ")[0] for n in CLASS_NAMES], fontsize=10)
ax.set_title("Training set class distribution", fontsize=11)
ax.set_ylabel("Count")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f"{count:,}\n({100*count/len(y_train):.1f}%)",
            ha="center", va="bottom", fontsize=8)

# Sample ECG beats per class
ax = axes[1]
t = np.arange(X_train.shape[1])
for c in range(5):
    idx = np.where(y_train == c)[0][0]
    ax.plot(t, X_train[idx] + c * 1.6, color=CLASS_COLORS[c],
            linewidth=1.2, label=CLASS_NAMES[c])
ax.set_title("Sample heartbeat per class (offset for clarity)", fontsize=11)
ax.set_xlabel("Sample")
ax.set_ylabel("Amplitude (offset)")
ax.legend(fontsize=8, loc="upper right")
ax.set_yticks([])

plt.tight_layout()
plt.show()

## 4 · Train TCRPClassifier (EXP-C02)

In [ ]:
# Experiment configuration (mirrors EXP-C02 in scripts/run_classification.py)
EXP_CONFIG = {
    "dataset": "MITBIH",
    "model": {
        "C": 5,        # number of classes
        "L": 25,       # segment length
        "stride": 5,   # segment stride
        "periods": [18, 36],  # periodic concept lags
        "k_max": 2,    # max ACF lag
    },
    "trainer": {
        "batch_size": 128,
        "max_epochs": 150,
        "es_patience": 20,
        "lr": 1e-3,
    },
    "weighted_sampling": True,  # balance imbalanced classes
}

SEED = 42
ADVERSARIAL = False   # set True in the adversarial cell below

CHECKPOINT_DIR = Path("/content/checkpoints")
RESULTS_DIR    = Path("/content/results")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Config:")
for k, v in EXP_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\ncheckpoints → {CHECKPOINT_DIR}")
print(f"results     → {RESULTS_DIR}")

In [ ]:
import time
from tcrp.dataset.classification_datasets import build_classification_loaders
from tcrp.model.classifier import TCRPClassConfig, TCRPClassifier
from tcrp.model.tcrp_forecaster.components.adversarial import AdversarialTCRPClassifier
from tcrp.training.classification_trainer import (
    AdversarialClassificationTrainer,
    ClassificationTrainer,
)
from tcrp.utils.misc import seed_everything

seed_everything(SEED)

run_name = f"EXP-C02_MITBIH_seed{SEED}" + ("_adv" if ADVERSARIAL else "")
ckpt_path = str(CHECKPOINT_DIR / f"{run_name}_best.pt")

# ── Data ────────────────────────────────────────────────────────────────────
loaders = build_classification_loaders(
    dataset_name=EXP_CONFIG["dataset"],
    batch_size=EXP_CONFIG["trainer"]["batch_size"],
    weighted_sampling=EXP_CONFIG["weighted_sampling"],
    data_root=str(DATA_ROOT),
)
print(f"train={loaders['n_train']:,}  val={loaders['n_val']:,}  test={loaders['n_test']:,}")

# ── Model ────────────────────────────────────────────────────────────────────
config = TCRPClassConfig(**EXP_CONFIG["model"])
base_model = TCRPClassifier(config)

if ADVERSARIAL:
    model = AdversarialTCRPClassifier(base_model=base_model, alpha=0.0)
else:
    model = base_model

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params={n_params:,}  K={config.K}  C={config.C}")

# ── Trainer ──────────────────────────────────────────────────────────────────
trainer_kw = EXP_CONFIG["trainer"]
trainer_cls = AdversarialClassificationTrainer if ADVERSARIAL else ClassificationTrainer
trainer = trainer_cls(
    model=model,
    config=config,
    lr=trainer_kw["lr"],
    es_patience=trainer_kw["es_patience"],
    checkpoint_path=ckpt_path,
    class_weights=loaders["class_weights"],
    device=DEVICE,
)

# ── Train ─────────────────────────────────────────────────────────────────────
t0 = time.time()
trainer.fit(loaders["train"], loaders["val"], max_epochs=trainer_kw["max_epochs"])
elapsed = time.time() - t0

print(f"\nTraining finished in {elapsed:.0f}s  →  {ckpt_path}")

## 5 · Evaluate on test set

In [ ]:
from tcrp.eval.classification_metrics import evaluate_all

# Load best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))

# Use base model for eval (unwrap adversarial wrapper if present)
eval_model = model.base if ADVERSARIAL else model

metrics = evaluate_all(
    eval_model,
    loaders["test"],
    device=DEVICE,
    class_weights=loaders["class_weights"],
)

print(f"{'='*50}")
print(f"  Accuracy   : {metrics['accuracy']:.4f}")
print(f"  Macro F1   : {metrics['macro_f1']:.4f}")
print(f"  Cross-Ent  : {metrics['cross_entropy']:.4f}")
print(f"{'='*50}")
print("\nPer-class accuracy:")
for c, acc in metrics["per_class_accuracy"].items():
    print(f"  Class {c} ({CLASS_NAMES[c]:<28s}): {acc:.4f}")

In [ ]:
# Confusion matrix
cm = metrics["confusion_matrix"]  # (5, 5) int64
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)  # row-normalised

short_names = [n.split(" — ")[0] for n in CLASS_NAMES]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Confusion Matrix (counts)"),
    (axes[1], cm_norm, ".2f",  "Confusion Matrix (row-normalised)"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=short_names, yticklabels=short_names,
        linewidths=0.5, linecolor="#e0e0e0", ax=ax,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True", fontsize=10)
    ax.set_title(title, fontsize=11)

plt.suptitle(f"MIT-BIH · {run_name}\naccuracy={metrics['accuracy']:.4f}  macro-F1={metrics['macro_f1']:.4f}",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{run_name}_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULTS_DIR / f'{run_name}_confusion.png'}")

## 6 · Concept profiles per arrhythmia class

For each AAMI class, we compute the **mean pooled concept activation** `h ∈ ℝᴷ` over all correctly predicted test samples.
Positive mean relevance → concept pushes toward that class; negative → pushes away.

In [ ]:
# Build concept name list matching the scorer's output order
BASE_CONCEPT_NAMES = [
    "trend_slope",      # 0
    "trend_r2",         # 1
    "monotonicity_up",  # 2
    "monotonicity_dn",  # 3
    "curvature_pos",    # 4
    "curvature_neg",    # 5
    "volatility",       # 6
    "volatility_ratio", # 7
    "stochasticity",    # 8
    "hurst",            # 9
    "skewness",         # 10
    "kurtosis",         # 11
    "jump_indicator",   # 12
    "break_score",      # 13
    "tendency",         # 14
    "autocorr_lag1",    # 15
    "acf_k1",           # 16
    "acf_k2",           # 17
]

concept_names = BASE_CONCEPT_NAMES + [f"period_{p}" for p in EXP_CONFIG["model"]["periods"]]
print(f"K = {config.K}  concept names ({len(concept_names)}):")
for i, n in enumerate(concept_names):
    print(f"  [{i:2d}] {n}")

In [ ]:
from tcrp.eval.concept_class_profile import class_concept_profiles

profiles = class_concept_profiles(
    eval_model, loaders["test"], concept_names=concept_names, device=DEVICE
)

# Convert to DataFrame (rows = concepts, cols = classes)
df_profiles = pd.DataFrame(profiles).T  # (n_classes, K)
df_profiles.index.name = "class"
df_profiles.columns = concept_names

print("Mean concept activation per correctly-predicted class:")
display(df_profiles.round(4))

In [ ]:
# Heatmap: concept × class
fig, ax = plt.subplots(figsize=(14, 4))

data = df_profiles.values  # (n_classes, K)
vmax = np.abs(data).max()

im = ax.imshow(data, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
fig.colorbar(im, ax=ax, label="Mean h_k", shrink=0.8)

ax.set_yticks(range(len(df_profiles)))
ax.set_yticklabels(
    [CLASS_NAMES[int(i)] for i in df_profiles.index],
    fontsize=9
)
ax.set_xticks(range(len(concept_names)))
ax.set_xticklabels(concept_names, rotation=45, ha="right", fontsize=8)
ax.set_title("Concept activation profiles per arrhythmia class", fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{run_name}_concept_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top-5 discriminative concepts per class
TOP_N = 5
print(f"Top-{TOP_N} concepts (by |mean activation|) per class\n" + "─" * 60)

for cls_idx in sorted(profiles.keys()):
    p = profiles[cls_idx]
    ranked = sorted(p.items(), key=lambda kv: abs(kv[1]), reverse=True)[:TOP_N]
    print(f"\nClass {cls_idx} · {CLASS_NAMES[cls_idx]}")
    for rank, (name, val) in enumerate(ranked, 1):
        direction = "+" if val >= 0 else "-"
        print(f"  {rank}. {direction} {name:<25s}  h={val:+.5f}")

In [ ]:
# Radar chart for the top discriminative concepts
# Select top-8 concepts by max absolute activation across all classes
max_act = df_profiles.abs().max(axis=0)
top8 = max_act.nlargest(8).index.tolist()

sub = df_profiles[top8]  # (n_classes, 8)
N = len(top8)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for cls_idx in sorted(profiles.keys()):
    values = sub.loc[cls_idx].tolist() + sub.loc[cls_idx].tolist()[:1]
    ax.plot(angles, values, linewidth=1.8, color=CLASS_COLORS[cls_idx],
            label=CLASS_NAMES[cls_idx])
    ax.fill(angles, values, alpha=0.08, color=CLASS_COLORS[cls_idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(top8, fontsize=9)
ax.set_title("Top-8 concept radar per AAMI class", fontsize=11, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{run_name}_radar.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · (Optional) Save results + checkpoints to Google Drive

In [ ]:
# Run this cell only if you want to persist to Drive
SAVE_TO_DRIVE = False   # ← set True to enable

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    import shutil
    drive_dir = Path("/content/drive/MyDrive/TCRP_results") / run_name
    drive_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(ckpt_path, drive_dir / Path(ckpt_path).name)
    for f in RESULTS_DIR.glob(f"{run_name}*"):
        shutil.copy(f, drive_dir / f.name)

    print(f"Saved to Google Drive: {drive_dir}")

## 8 · (Optional) Adversarial training (GRL)

Re-trains with the Gradient Reversal Layer to enforce concept purity — the encoder is penalised for encoding concept-irrelevant information.
Alpha ramps from 0 → 1 after a warmup period using the DANN sigmoid schedule.

Expected outcome: slightly lower accuracy (trade-off), but more interpretable concept-class alignment.

In [ ]:
seed_everything(SEED)

adv_config = TCRPClassConfig(
    **EXP_CONFIG["model"],
    alpha_max=1.0,
    warmup_epochs=20,
)
adv_base  = TCRPClassifier(adv_config)
adv_model = AdversarialTCRPClassifier(base_model=adv_base, alpha=0.0)

adv_run_name = f"EXP-C02_MITBIH_seed{SEED}_adv"
adv_ckpt     = str(CHECKPOINT_DIR / f"{adv_run_name}_best.pt")

adv_trainer = AdversarialClassificationTrainer(
    model=adv_model,
    config=adv_config,
    lr=EXP_CONFIG["trainer"]["lr"],
    es_patience=EXP_CONFIG["trainer"]["es_patience"],
    checkpoint_path=adv_ckpt,
    class_weights=loaders["class_weights"],
    device=DEVICE,
)

t0 = time.time()
adv_trainer.fit(loaders["train"], loaders["val"],
                max_epochs=EXP_CONFIG["trainer"]["max_epochs"])
print(f"\nAdversarial training finished in {time.time()-t0:.0f}s")

In [ ]:
# Evaluate and compare standard vs adversarial
adv_model.load_state_dict(torch.load(adv_ckpt, map_location=DEVICE, weights_only=True))
adv_metrics = evaluate_all(
    adv_model.base, loaders["test"], device=DEVICE, class_weights=loaders["class_weights"]
)

print(f"{'':30s} {'Standard':>10} {'Adversarial':>12}")
print("-" * 55)
for key in ["accuracy", "macro_f1", "cross_entropy"]:
    print(f"  {key:<28s} {metrics[key]:>10.4f} {adv_metrics[key]:>12.4f}")

## 9 · Summary

In [ ]:
import json

summary = {
    "experiment":     "EXP-C02",
    "dataset":        "MIT-BIH",
    "seed":           SEED,
    "adversarial":    False,
    "K":              config.K,
    "C":              config.C,
    "n_params":       n_params,
    "accuracy":       round(metrics["accuracy"], 4),
    "macro_f1":       round(metrics["macro_f1"], 4),
    "cross_entropy":  round(metrics["cross_entropy"], 4),
    "per_class_accuracy": {str(k): round(v, 4) for k, v in metrics["per_class_accuracy"].items()},
    "confusion_matrix":   metrics["confusion_matrix"].tolist(),
    "checkpoint":     ckpt_path,
}

out_json = RESULTS_DIR / f"{run_name}_summary.json"
out_json.write_text(json.dumps(summary, indent=2))
print(json.dumps({k: v for k, v in summary.items() if k != "confusion_matrix"}, indent=2))
print(f"\nFull summary saved → {out_json}")

## 10 · Baseline Comparison

Train five standard time-series classifiers on the same MIT-BIH split and compare against TCRP (EXP-C02).

| Model | Architecture |
|---|---|
| **MLP** | Flatten → 3 × (Linear 500 → ReLU → Dropout) |
| **LSTM** | Bidirectional LSTM (2 layers) → linear head |
| **FCN** | Conv(128,k=8) → Conv(256,k=5) → Conv(128,k=3) + GAP |
| **ResNet** | 3 residual blocks (1→64→128→128) + GAP |
| **N-BEATS** | N-BEATS stack with backcast residuals, accumulated logits |

All baselines use weighted cross-entropy (same class weights as TCRP) and identical early stopping.

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from tcrp.model.classification_baselines import BASELINE_MODELS, build_baseline_classifier
from tcrp.eval.classification_metrics import evaluate_all
from tcrp.utils.misc import seed_everything

# MIT-BIH fixed dimensions
T_MITBIH = 187
C_MITBIH = 5

BL_TRAINER_CFG = {
    "batch_size": 128,   # same as EXP-C02
    "max_epochs": 150,
    "es_patience": 20,
    "lr": 1e-3,
}


def _train_baseline_mitbih(model_type: str, seed: int = SEED) -> dict:
    """Train one baseline on the already-loaded MIT-BIH loaders and return metrics."""
    seed_everything(seed)

    run_name = f"BL-{model_type.upper()}_EXP-C02_seed{seed}"
    ckpt_path = str(CHECKPOINT_DIR / f"{run_name}_best.pt")

    model = build_baseline_classifier(model_type, T=T_MITBIH, C=C_MITBIH).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  params={n_params:,}")

    optimizer = Adam(model.parameters(), lr=BL_TRAINER_CFG["lr"])
    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
    class_w   = loaders["class_weights"].to(DEVICE)

    best_val_ce    = float("inf")
    epochs_no_impr = 0

    print(f"\n{'Epoch':>6} | {'train_CE':>10} | {'val_CE':>10} | {'val_acc':>8} | {'lr':>10}")
    print("-" * 56)

    t0 = time.time()
    for epoch in range(1, BL_TRAINER_CFG["max_epochs"] + 1):
        # ── train ────────────────────────────────────────────────────────────
        model.train()
        tot, cnt = 0.0, 0
        for x, y in loaders["train"]:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(x).y_hat, y, weight=class_w)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tot += loss.item() * x.shape[0]
            cnt += x.shape[0]
        train_ce = tot / cnt

        # ── validate ─────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            v_ce, v_correct, v_cnt = 0.0, 0, 0
            for x, y in loaders["val"]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x).y_hat
                v_ce      += F.cross_entropy(logits, y, reduction="sum").item()
                v_correct += (logits.argmax(dim=-1) == y).sum().item()
                v_cnt     += x.shape[0]
        val_ce  = v_ce / v_cnt
        val_acc = v_correct / v_cnt

        scheduler.step(val_ce)
        lr = optimizer.param_groups[0]["lr"]
        print(f"{epoch:6d} | {train_ce:10.6f} | {val_ce:10.6f} | {val_acc:8.4f} | {lr:10.2e}")

        if val_ce < best_val_ce:
            best_val_ce    = val_ce
            epochs_no_impr = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_impr += 1

        if epochs_no_impr >= BL_TRAINER_CFG["es_patience"]:
            print(f"Early stopping at epoch {epoch}.")
            break

    elapsed = time.time() - t0

    # ── test evaluation ───────────────────────────────────────────────────────
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    m = evaluate_all(model, loaders["test"], device=DEVICE, class_weights=loaders["class_weights"])

    print(f"\n  accuracy={m['accuracy']:.4f}  macro_F1={m['macro_f1']:.4f}  elapsed={elapsed:.0f}s")
    return {
        "model":      model_type,
        "n_params":   n_params,
        "accuracy":   m["accuracy"],
        "macro_f1":   m["macro_f1"],
        "cross_entropy": m["cross_entropy"],
        "per_class_accuracy": m["per_class_accuracy"],
        "confusion_matrix":   m["confusion_matrix"],
        "elapsed_s":  round(elapsed, 1),
        "checkpoint": ckpt_path,
    }

print(f"Baseline models to run: {list(BASELINE_MODELS)}")
print(f"T={T_MITBIH}  C={C_MITBIH}  device={DEVICE}")

In [ ]:
# Train all baseline models sequentially
# Each model trains for up to 150 epochs (~2-5 min each on T4 GPU)

baseline_results = {}

for model_type in BASELINE_MODELS:
    print(f"\n{'='*66}")
    print(f"  BL-{model_type.upper()}  (MIT-BIH  T=187  C=5)")
    print(f"{'='*66}")
    baseline_results[model_type] = _train_baseline_mitbih(model_type, seed=SEED)

print("\nAll baselines complete.")

In [ ]:
# Comparison table: TCRP vs all baselines
rows = []

# TCRP (from section 5)
rows.append({
    "Model":    "TCRP (EXP-C02)",
    "Params":   f"{n_params:,}",
    "Accuracy": f"{metrics['accuracy']:.4f}",
    "Macro F1": f"{metrics['macro_f1']:.4f}",
    "Cross-Ent": f"{metrics['cross_entropy']:.4f}",
})

# Baselines
for mt, r in baseline_results.items():
    rows.append({
        "Model":    f"BL-{mt.upper()}",
        "Params":   f"{r['n_params']:,}",
        "Accuracy": f"{r['accuracy']:.4f}",
        "Macro F1": f"{r['macro_f1']:.4f}",
        "Cross-Ent": f"{r['cross_entropy']:.4f}",
    })

df_cmp = pd.DataFrame(rows).set_index("Model")
print("MIT-BIH Classification — Model Comparison\n")
display(df_cmp)

In [ ]:
# Grouped bar chart: accuracy and macro-F1 for all models
model_labels = ["TCRP"] + [f"BL-{mt.upper()}" for mt in BASELINE_MODELS]
accs  = [metrics["accuracy"]] + [baseline_results[mt]["accuracy"]  for mt in BASELINE_MODELS]
f1s   = [metrics["macro_f1"]] + [baseline_results[mt]["macro_f1"]  for mt in BASELINE_MODELS]

x     = np.arange(len(model_labels))
width = 0.35
colors_acc = ["#4c72b0"] + ["#aec7e8"] * len(BASELINE_MODELS)
colors_f1  = ["#c44e52"] + ["#f7b6b6"] * len(BASELINE_MODELS)

fig, ax = plt.subplots(figsize=(12, 5))
bars_acc = ax.bar(x - width/2, accs, width, label="Accuracy", color=colors_acc, edgecolor="white")
bars_f1  = ax.bar(x + width/2, f1s,  width, label="Macro F1",  color=colors_f1,  edgecolor="white")

# Value labels
for bar in bars_acc:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
for bar in bars_f1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=10)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("MIT-BIH Arrhythmia Classification — TCRP vs Baselines", fontsize=12)
ax.legend(fontsize=10)
ax.axhline(y=accs[0], color="#4c72b0", linewidth=0.8, linestyle="--", alpha=0.5)
ax.axhline(y=f1s[0],  color="#c44e52", linewidth=0.8, linestyle="--", alpha=0.5)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "mitbih_baseline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {RESULTS_DIR / 'mitbih_baseline_comparison.png'}")

# Per-class F1 breakdown (TCRP vs best baseline)
best_bl_name = max(BASELINE_MODELS, key=lambda mt: baseline_results[mt]["macro_f1"])
best_bl      = baseline_results[best_bl_name]

print(f"\nBest baseline: BL-{best_bl_name.upper()} (macro-F1={best_bl['macro_f1']:.4f})")
print(f"\nPer-class accuracy  |  TCRP  vs  BL-{best_bl_name.upper()}")
print("-" * 60)
for c in range(C_MITBIH):
    tcrp_ca = metrics["per_class_accuracy"].get(c, 0.0)
    bl_ca   = best_bl["per_class_accuracy"].get(c, 0.0)
    delta   = bl_ca - tcrp_ca
    arrow   = "▲" if delta > 0 else ("▼" if delta < 0 else " ")
    print(f"  Class {c} ({CLASS_NAMES[c]:<28s})  {tcrp_ca:.4f}  {bl_ca:.4f}  {arrow}{abs(delta):.4f}")